In [4]:
#Loading Pdf
from langchain_community.document_loaders.pdf import PyPDFLoader
pdf_loader=PyPDFLoader(r"C:\Users\HP\Downloads\Python\PracticeQuestions\Datasets\IRass.pdf")

document=pdf_loader.load()

print(document)


C:\Users\HP\AppData\Local\Temp\ipykernel_14600\3700056092.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders.pdf import PyPDFLoader
e:\DeepLearning\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


[Document(metadata={'producer': 'www.ilovepdf.com', 'creator': 'Microsoft® Word 2016', 'creationdate': '2026-09-05T13:52:14+00:00', 'author': 'SIERD training', 'moddate': '2026-09-05T13:52:15+00:00', 'source': 'C:\\Users\\HP\\Downloads\\Python\\PracticeQuestions\\Datasets\\IRass.pdf', 'total_pages': 2, 'page': 0, 'page_label': '1'}, page_content='Assignment – Module 1: Information Retrieval \nTotal Marks: 30 \nInstructions \n\uf0b7 Write your Name and Roll No. on the first page. \n\uf0b7 Take clear photographs/scans of all pages and combine them into one PDF .  \n\uf0b7 Upload the PDF on Google Classroom.  \n\uf0b7 Also submit the handwritten assignment.  \n\uf0b7 Use the prescribed book as the primary reference. PPTs are only for understanding.  \n\uf0b7 Follow the book’s notations, terminology, and standard conventions.  \n\uf0b7 Show all intermediate steps wherever required.  \n \nQ1. Consider the following document collection:          6 Marks \n\uf0b7 D1: information retrieval use

In [5]:
#Chunks Breaking
from langchain_text_splitters import RecursiveCharacterTextSplitter

def split_doc(document,chunk_size=200,chunk_overlap=50):
    text_splitter=RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap
    )
    chunked_docs=text_splitter.split_documents(document)
    return chunked_docs
    
chunks=split_doc(document)
print(chunks)

[Document(metadata={'producer': 'www.ilovepdf.com', 'creator': 'Microsoft® Word 2016', 'creationdate': '2026-09-05T13:52:14+00:00', 'author': 'SIERD training', 'moddate': '2026-09-05T13:52:15+00:00', 'source': 'C:\\Users\\HP\\Downloads\\Python\\PracticeQuestions\\Datasets\\IRass.pdf', 'total_pages': 2, 'page': 0, 'page_label': '1'}, page_content='Assignment – Module 1: Information Retrieval \nTotal Marks: 30 \nInstructions \n\uf0b7 Write your Name and Roll No. on the first page.'), Document(metadata={'producer': 'www.ilovepdf.com', 'creator': 'Microsoft® Word 2016', 'creationdate': '2026-09-05T13:52:14+00:00', 'author': 'SIERD training', 'moddate': '2026-09-05T13:52:15+00:00', 'source': 'C:\\Users\\HP\\Downloads\\Python\\PracticeQuestions\\Datasets\\IRass.pdf', 'total_pages': 2, 'page': 0, 'page_label': '1'}, page_content='\uf0b7 Take clear photographs/scans of all pages and combine them into one PDF .  \n\uf0b7 Upload the PDF on Google Classroom.  \n\uf0b7 Also submit the handwritten 

In [12]:
#Create Embeddings
from sentence_transformers import SentenceTransformer

class EmbeddingManager:
    def __init__(self,model_name="all-MiniLM-L6-v2"):
        self.model_name=model_name
        self.model=SentenceTransformer(self.model_name)
        
    def embeddingGenerator(self,text):
        embdeddings=self.model.encode(
            text,show_progress_bar=True
        )
        return embdeddings

embedding_manager=EmbeddingManager()

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 4378.85it/s]


In [24]:
# VectorStore
import uuid
import chromadb
import os

class VectorStoreManager:
    def __init__(self,persist_dict="project/vector_store",collection_name="docs"):
        self.collection_name=collection_name
        self.persist_dict=persist_dict
        self.collection=None
        self.client=None
        
        self. _initalize_store()
            
    def _initalize_store(self):
        os.makedirs(self.persist_dict,exist_ok=True)
        
        self.client=chromadb.PersistentClient(
            path=self.persist_dict
        )
        self.collection=self.client.get_or_create_collection(
            name=self.collection_name,
            metadata={
                "description":"This is my project on IR Assignment"
            }
        )
        
        print(self.collection.count())
        
    def add_documents(self,documents,embeddings):
        if(len(documents)!=len(embeddings)):
            raise ValueError("Lengths are not equal")
        
        ids=[]
        all_metadata=[]
        document_content=[]
        embedding_list=[]
        
        for i, (doc,embedding) in enumerate(zip(documents,embeddings)):
            doc_id=f"The doc id is {uuid.uuid4()}"
            ids.append(doc_id)
            metadata=dict(doc.metadata)
            all_metadata.append(metadata)
            document_content.append(doc.page_content)
            embedding_list.append(embedding.tolist())
            
        self.collection.add(
                ids=ids,
                metadatas=all_metadata,
                documents=document_content,
                embeddings=embedding_list
            )
            
        print("Total doc to be added",len(document_content))
        
vector_store=VectorStoreManager()

texts=[doc.page_content for doc in chunks]

embedding=embedding_manager.embeddingGenerator(texts)

        
vector_store.add_documents(
        chunks,embedding
    )
        


0


Batches: 100%|██████████| 1/1 [00:00<00:00,  3.45it/s]

Total doc to be added 18


In [32]:
class RAGRetrieval:

    def __init__(self, embedding_manager, vector_store):

        self.embedding_manager = embedding_manager
        self.vector_store=vector_store

    def retrieval(self, query, top_k, score_threshold=0.0):

        # 1. Query → embedding
        query_embedding = self.embedding_manager.embeddingGenerator([query])[0]

        # 2. Search vector store
        results = self.vector_store.collection.query(
            query_embeddings=[query_embedding.tolist()],
            n_results=top_k
        )

        retrieved_docs = []

        if results["documents"] and results["documents"][0]:

            ids = results["ids"][0]
            metadatas = results["metadatas"][0]
            documents = results["documents"][0]
            distances = results["distances"][0]

            # 3. Process results
            for i, (doc_id, metadata, document, distance) in enumerate(
                zip(ids, metadatas, documents, distances)
            ):

                similarity_score = 1 - distance

                if similarity_score >= score_threshold:

                    retrieved_docs.append({
                        "id": doc_id,
                        "document": document,
                        "metadata": metadata,
                        "distance": distance,
                        "similarity_score": similarity_score,
                        "rank": i + 1
                    })

        else:
            print("No documents found")

        return retrieved_docs

In [33]:
ragretriever = RAGRetrieval(
    embedding_manager,
    vector_store
)

results = ragretriever.retrieval(
    "information retrieval",
    top_k=3
)

print(results)

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches: 100%|██████████| 1/1 [00:00<00:00,  5.48it/s]


[{'id': 'The doc id is 5afe5984-4e27-4544-ac96-3bb0866adf57', 'document': '\uf0b7 D3: efficient information retrieval  \n(a) Construct the positional index for information, retrieval, and systems. [3] \n(b) Demonstrate how the phrase query: \n"information retrieval"', 'metadata': {'creationdate': '2026-09-05T13:52:14+00:00', 'total_pages': 2, 'source': 'C:\\Users\\HP\\Downloads\\Python\\PracticeQuestions\\Datasets\\IRass.pdf', 'page_label': '1', 'page': 0, 'moddate': '2026-09-05T13:52:15+00:00', 'producer': 'www.ilovepdf.com', 'creator': 'Microsoft® Word 2016', 'author': 'SIERD training'}, 'distance': 0.5881369709968567, 'similarity_score': 0.4118630290031433, 'rank': 1}, {'id': 'The doc id is 8c8f8624-1ba3-4903-99bd-d2cf6ae3d774', 'document': '"information retrieval" \nis processed using the positional index. [2]', 'metadata': {'producer': 'www.ilovepdf.com', 'creator': 'Microsoft® Word 2016', 'source': 'C:\\Users\\HP\\Downloads\\Python\\PracticeQuestions\\Datasets\\IRass.pdf', 'creat